In [1]:
import numpy as np
import numpy_extended as npe
import pickle
from scipy import io as sio
import os

In [2]:
import matplotlib.pyplot as plt

In [2]:
from minimal_winding_model import * 

PHCv2.4.86 released 2022-05-30 works!


In [5]:
%matplotlib

Using matplotlib backend: QtAgg


In [3]:
char_poly = minimal_winding_char_poly()

# Calculation for x major

In [6]:
data = sio.loadmat("data/minimal_winding_aGBZ.mat")

In [7]:
aGBZ_params = data['params']

In [8]:
with open("data/minimal_winding_aGBZ.pkl", "rb") as fp:
    aGBZ_points = pickle.load(fp)

In [9]:
print(len(aGBZ_points[0]))

8


In [9]:
curr_ptr = 10
curr_param = aGBZ_params[curr_ptr,:]
curr_point = aGBZ_points[curr_ptr]

In [13]:
print(curr_param)
print(curr_point)

[ 0.77908057+0.62692381j -0.6506183 +0.75940492j]
[[ 2.66972350e+000-2.23887943e-010j  1.14220424e+000-4.02497244e-001j
  -1.02518420e-001-1.32877883e+000j  1.07578154e+000-7.86674831e-001j]
 [-1.93072557e+000-8.19099455e-001j -1.01424213e+000+1.31381630e-001j
  -1.00511979e+000-2.43807931e-001j -3.21837706e-001-9.82918409e-001j]
 [ 1.01192634e+000-3.92671064e-011j  7.44580508e-001-2.62380048e-001j
   5.39577398e-002+6.99366046e-001j -5.66207907e-001+4.14044575e-001j]
 [-6.26580639e+063-3.36224271e+064j  4.88694865e-112+2.51640015e-112j
  -1.15503232e-111+1.45308968e-111j  6.25563743e-112-3.81510697e-112j]
 [-1.93072557e+000+8.19099454e-001j -8.72542615e-001+5.33495662e-001j
   9.55826574e-001-3.95099567e-001j  8.39098273e-001+6.04667013e-001j]
 [-2.37312999e+000-2.25823698e-010j -2.03122861e+000+7.15777346e-001j
  -3.15119484e+000+2.43122114e-001j  1.86559690e+000+2.55121255e+000j]
 [ 1.40836554e+000+7.12654520e-001j  1.51736964e+000+8.32833324e-001j
  -2.10913800e+000+8.58695247e-001

In [15]:
print(char_poly.num.to_str(csolver.CStrVec(['E', 'x', 'y'])))
print(char_poly.denom_orders)

+(-1+0*i)*(y**2)+(-1+0*i)*x*y+(-1+0*i)*(x**3)+(-1+0*i)*(x**3)*y+(1+0*i)*E*(x**2)*y;
aGBZ_2D_solver_IndexVec[0, 2, 1]


In [11]:
winding_solver = csolver.GBZConditionChecker2D(curr_point[0, :3], curr_param[1], char_poly, 1, 0.1)

In [15]:
print(curr_point[3,:])

[-6.26580639e+063-3.36224271e+064j  4.88694865e-112+2.51640015e-112j
 -1.15503232e-111+1.45308968e-111j  6.25563743e-112-3.81510697e-112j]


In [16]:
for j in [0]:
    print(j)
    winding_solver = csolver.GBZConditionChecker2D(curr_point[j, :3], curr_param[1], char_poly, 1, 0.1)
    vals, params = winding_solver.eval_param_path(curr_point[j,1] * np.exp(1j * np.linspace(0, 2 * np.pi, 200)), curr_point[j,[0,2]],
                                                N_points_loop=200, N_points_detour=50, theta_detour=1e-2*np.pi)

0


In [19]:
vals2, params2 = winding_solver.eval_param_path(curr_point[j,1] * np.exp(1j * np.linspace(0, 2 * np.pi, 200)), params[-1],
                                            N_points_loop=200, N_points_detour=50, theta_detour=1e-2*np.pi)

In [12]:
vals1, params1 = winding_solver.eval_param_path(curr_point[0,1] * np.exp(1j * np.linspace(0, 2 * np.pi, 200)), curr_point[0,[0, 2]],
                                            N_points_loop=200, N_points_detour=50, theta_detour=1e-2*np.pi)
vals2, params2 = winding_solver.eval_param_path(curr_point[0,1] * np.exp(1j * np.linspace(0, 2 * np.pi, 200)), params1[-1],
                                            N_points_loop=200, N_points_detour=50, theta_detour=1e-2*np.pi)

In [12]:
N_points_loop = 200
N_points_detour = 50
theta_detour = 1e-3 * np.pi
zero_val_th = 1e-3
eval_fun = lambda beta_path, start_param: winding_solver.eval_param_path(
    beta_path, start_param, 
    theta_detour = theta_detour, N_points_loop=N_points_loop, N_points_detour = N_points_detour, 
    zero_eps=zero_val_th
)

In [13]:
beta0 = curr_point[0,1]
circular_loop = beta0 * np.exp(1j * np.linspace(0, 2 * np.pi, N_points_loop))

In [14]:
vals1, params1 = eval_fun(circular_loop, curr_point[0, [0,2]])

In [15]:
vals2, params2 = eval_fun(circular_loop, params1[-1])

In [16]:
result = csolver.winding_phase_auto_detour_path_check(
    curr_point[0,1], theta_detour, zero_val_th, N_points_loop, N_points_detour, eval_fun, 
    curr_point[0,[0,2]], 0.1
)

0
0 finishes
1


In [16]:
print(result)
# print(len(result))

([(1.1399425195654302-0.40885890938225256j), (1.1404081050575916-0.409007593271071j), (1.1408822753612267-0.40912608070692685j), (1.1413630742098686-0.4092138828506924j), (1.1418485179899127-0.4092706374603878j), (1.1423366039243368-0.40929611038566877j), (1.1428253183354828-0.40929019653385207j), (1.14331264495281-0.40925292030349225j), (1.143796573231345-0.40918443548372135j), (1.1442751066465138-0.40908502461976715j), (1.1447462709311247-0.4089550978472669j), (1.1452081222205293-0.4087951912001863j), (1.1456587550723505-0.4086059643993243j), (1.1460963103276953-0.4083881981305282j), (1.1465189827814142-0.4081427908238474j), (1.1469250286297696-0.4078707549469149j), (1.1473127726647798-0.40757321282784736j), (1.1476806151855614-0.40725139202489924j), (1.1480270385981548-0.4069066202619714j), (1.1483506136766037-0.4065403199508715j), (1.1486500054594593-0.40615400232292304j), (1.1489239787573795-0.40574926119413585j), (1.149171403249103-0.40532776638965945j), (1.1493912581447727-0.404

In [17]:
outer_loop, inner_loop = result
outer_loop = np.array(outer_loop)
inner_loop = np.array(inner_loop)

In [19]:
plt.plot(outer_loop.real, outer_loop.imag)
plt.plot(inner_loop.real, inner_loop.imag)
plt.plot(beta0.real, beta0.imag, 'x')
plt.plot((beta0*curr_param[0]).real, (beta0*curr_param[0]).imag, 'x')

(1.142204239262712-0.40249724409580206j)

# plot result

In [7]:
with open("data/minimal_model_test.pkl", "rb") as fp:
    data = pickle.load(fp)

In [28]:
all_segments, GBZ_points, GBZ_params, non_GBZ_points, non_GBZ_params = data

In [10]:
for seg in all_segments:
    plt.plot(seg[:,0].real, seg[:,0].imag, '.')

In [24]:
fig = plt.figure()
ax= fig.add_subplot(projection='3d')
for seg in all_segments:
    ax.plot(seg[:,1].real, seg[:,1].imag, seg[:,0].real)

In [15]:
fig = plt.figure()
ax = fig.gca()
for j in range(50):
    if(os.path.exists("data/GBZ_data_6/x_major_%d.pkl"%(j))):
        with open("data/GBZ_data_6/x_major_%d.pkl"%(j), "rb") as fp:
            all_segments = pickle.load(fp)
        for seg in all_segments:
            ax.plot(seg[:,0].real, seg[:,0].imag)
plt.show()

# Y major debug

In [22]:
print(char_poly.num.to_str(csolver.CStrVec(['x', 'y', 'z'])))

+(-1+0*i)*(z**2)+(-1+0*i)*y*z+(-0.100000000000000006+0*i)*y*(z**2)+(-1+0*i)*(y**3)+(-1+0*i)*(y**3)*z+(-0.100000000000000006+0*i)*(y**4)+(1+0*i)*x*(y**2)*z;


In [23]:
which_var = 2

In [24]:
with open("data/minimal_model_test.pkl", "rb") as fp:
    data = pickle.load(fp)

In [25]:
all_segments, GBZ_points, GBZ_params, non_GBZ_points, non_GBZ_params = data

In [11]:
print(len(all_segments))

4


In [12]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
for seg in all_segments:
    ax.plot(seg[:,2].real, seg[:,2].imag, seg[:,0].real)
    plt.pause(1)

In [14]:
check_result =csolver.check_GBZ_condition(char_poly, non_GBZ_points[0][:3], non_GBZ_params[0], 2)

[-0.03355522-0.01039632j  0.72733093-1.54099378j]
[-0.03355522-0.01039632j  0.72733093-1.54099378j]
[-0.03355522-0.01039632j  0.72733093-1.54099378j]
[-0.03355522-0.01039632j  0.72733093-1.54099378j]
[-0.03355522-0.01039632j  0.72733093-1.54099378j]
[-0.03355522-0.01039632j  0.72733093-1.54099378j]
[-0.03336102-0.02124416j  0.74296778-1.53334216j]
[-0.03352175-0.02124975j  0.74305715-1.53351742j]
[-0.03352189-0.02124534j  0.74305318-1.53351941j]
[-0.03352182-0.02124534j  0.74305315-1.53351937j]
[-0.03352182-0.02124534j  0.74305315-1.53351937j]
[-0.03355522-0.01039632j  0.72733093-1.54099378j]
[-0.03352182-0.02124534j  0.74305315-1.53351937j]
[-0.03352182-0.02124534j  0.74305315-1.53351937j]
[-0.03330472-0.03209044j  0.75862205-1.52573383j]
[-0.03346541-0.03209565j  0.75871235-1.52590856j]
[-0.03346552-0.03209124j  0.75870839-1.52591057j]
[-0.03346545-0.03209124j  0.75870837-1.52591053j]
[-0.03346545-0.03209124j  0.75870837-1.52591053j]
[-0.03352182-0.02124534j  0.74305315-1.53351937j]


Distances: [3.2357770689842407]


In [34]:
winding_solver = csolver.GBZConditionChecker2D(non_GBZ_points[0][:3], non_GBZ_params[0][0], char_poly, 2, 0.1)

In [28]:
beta0 = non_GBZ_points[0][2]

In [36]:
print(abs(beta0))

5.052090021422052


In [19]:
winding_solver.eval_param_path(beta0*np.exp(1j * np.linspace(0, 2 * np.pi, 100)), non_GBZ_points[0][[0,1]], 
                    N_points_loop = 200, N_points_detour = 50, theta_detour=5e-3*np.pi, zero_eps = 1e-3)

1,-1
1,-1
1,-1
1,-1
1,-1
1,-1
1,-1
1,-1
1,-1
1,-1
1,-1
1,-1
1,-1
1,-1
1,-1
1,-1
1,-1
1,-1
1,-1
1,-1
1,-1
1,-1
1,-1
1,-1
1,-1
1,-1
1,-2
1,-2
1,-2
1,-2
1,-2
0,-2
1,-1
0,-2
0,-2


Distances: [3.2515546367139727]


In [37]:
def ymajor_aGBZ_sol(phi1, beta2):
    expiphi1 = np.exp(-1j*phi1)
    eq_poly = np.array([1 + 1/beta2, 0, - expiphi1, - beta2 * expiphi1*(1 + expiphi1)])
    return np.roots(eq_poly)

In [49]:
def ymajor_GBZ_check(beta1, beta2):
    GBZ_conditions = (abs(beta2)/(beta1**2 * ( 1 + 1/beta2))) < abs(beta1)
    if(not GBZ_conditions):
        E = get_E(beta1, beta2)
        beta1_eq = [1 + 1/beta2, -E, 1, beta2]
        print(np.abs(np.roots(beta1_eq)))
    return GBZ_conditions

In [39]:
def get_E(beta1, beta2):
    return beta1 + 1/beta1 + beta1/beta2 + beta2/(beta1**2)

In [26]:
expiphi1 = non_GBZ_params[0][0]
phi1 = np.log(expiphi1).imag % (2 * np.pi)

In [41]:
print(phi1)

2.2791946702514188


In [29]:
beta2_list = beta0 * np.exp(1j * np.linspace(0, 2 * np.pi, 100))
all_sols = np.zeros((len(beta2_list), 3), dtype=complex)
for j in range(len(beta2_list)):
    all_sols[j, :] = ymajor_aGBZ_sol(phi1, beta2_list[j])

NameError: name 'ymajor_aGBZ_sol' is not defined

In [49]:
phi1

2.2791946702514188

In [50]:
GBZ_conditions = np.zeros_like(all_sols, dtype=bool)
conditions_winding_solver = np.zeros_like(all_sols, dtype=bool)
for j in range(len(beta2_list)):
    for k in range(3):
        beta1 = all_sols[j,k]
        beta2 = beta2_list[j]
        GBZ_conditions[j,k] = ymajor_GBZ_check(all_sols[j,k], beta2_list[j])
        # print(char_poly.eval(csolver.CScalarVec([get_E(beta1, beta2), beta1*np.exp(1j*phi1), beta2])))
        conditions_winding_solver[j,k] = winding_solver.GBZ_condition_1d([get_E(beta1, beta2), beta1], beta2, theta_detour=0.1*np.pi)
    #     beta1_list = beta1 * np.exp(1j*np.linspace(0, 2*np.pi, 200))
    #     val_list = np.zeros_like(beta1_list, dtype=complex)
    #     for u, curr_beta1 in enumerate(beta1_list):
    #         val_list[u] = char_poly.eval(csolver.CScalarVec([get_E(beta1, beta2), curr_beta1, beta2]))
    #     plt.plot(val_list.real, val_list.imag)
    #     break
    # break
        # input()

[2.0473 1.4353 1.4353]
[2.0661 1.4291 1.4291]
[2.0852 1.4234 1.4234]
[2.1045 1.418  1.418 ]
[2.124 1.413 1.413]
[2.1438 1.4084 1.4084]
[2.1637 1.4043 1.4043]
[2.1838 1.4005 1.4005]
[2.204  1.3971 1.3971]
[2.2243 1.3942 1.3942]
[2.2446 1.3916 1.3916]
[2.265  1.3895 1.3895]
[2.2854 1.3878 1.3878]
[2.3057 1.3866 1.3866]
[2.3259 1.3858 1.3858]
[2.3459 1.3854 1.3854]
[2.3657 1.3855 1.3855]
[2.3853 1.386  1.386 ]
[2.4046 1.387  1.387 ]
[2.4234 1.3885 1.3885]
[2.4419 1.3904 1.3904]
[2.4597 1.3929 1.3929]
[2.477  1.3958 1.3958]
[2.4936 1.3991 1.3991]
[2.5094 1.403  1.403 ]
[2.5243 1.4074 1.4074]
[2.5383 1.4123 1.4123]
[2.5511 1.4176 1.4176]
[2.5628 1.4235 1.4235]
1,-1
0,-2
0,-2
1,-1
0,-2
0,-2
1,-1
0,-2
0,-2
1,-1
0,-2
0,-2
1,-1
0,-2
0,-2
1,-1
0,-2
0,-2
1,-1
0,-2
0,-2
1,-1
0,-2
0,-2
1,-1
0,-2
0,-2
1,-1
0,-2
0,-2
1,-1
0,-2
0,-2
1,-1
0,-2
0,-2
1,-1
0,-2
0,-2
1,-1
0,-2
0,-2
1,-1
0,-2
0,-2
1,-1
0,-2
0,-2
1,-1
0,-2
0,-2
1,-1
0,-2
0,-2
1,-1
0,-2
0,-2
1,-1
0,-2
0,-2
1,-1
0,-2
0,-2
1,-1
0,-2
0,-2
1,-1
1

In [53]:
plt.figure()
for k in range(len(beta2_list)):
    for j in range(3):
        if(GBZ_conditions[k,j]):
            plt.plot(all_sols[k,j].real, all_sols[k,j].imag, 'b.')
        else:
            plt.plot(all_sols[k,j].real, all_sols[k,j].imag, 'r.')
plt.xlabel("$\\beta_1$, real")
plt.ylabel("$\\beta_1$, imag")
    # plt.pause(1)

Text(0, 0.5, '$\\beta_1$, imag')

In [64]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
for k in range(len(beta2_list)):
    for j in range(3):
        if(GBZ_conditions[k, j]):
            plt.plot(all_sols[k, j].real, all_sols[k, j].imag, k, '.')

In [30]:
new_solver = csolver.DirectGBZConditionChecker(char_poly, 2)

In [31]:
GBZ_points[0]

array([-0.0336-0.0104j,  0.7273-1.541j ,  5.0498+0.1528j, -0.8172+1.8003j])

In [37]:
# for j in range(len(non_GBZ_points)):
curr_j = 0
for phi in np.linspace(np.pi/101, np.pi * 100/101, 100):
    all_sols = new_solver.get_aGBZ_solutions(non_GBZ_points[curr_j][:3], 
                # non_GBZ_params[0], 
                [np.exp(1j * phi), 1])
    print(len(all_sols))
    plt.plot(all_sols[:,1].real, all_sols[:,1].imag, '.')
    # for j in range(all_sols.shape[0]):
    #     plt.plot(all_sols[j,0].real, all_sols[j,0].imag,'.')
    #     plt.plot(all_sols[j,0].real, all_sols[j,0].imag,'.')

2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
0


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [14]:
print(char_poly.denom_orders)

aGBZ_2D_solver_IndexVec[0, 2, 1]


In [22]:
two_eq = [
    new_solver.eff_poly.to_str(csolver.CStrVec(['x','y'])),
    new_solver.another_poly.to_str(csolver.CStrVec(['x', 'y']))
]

In [23]:
print(two_eq)

['+(-1+0*i)*(y**2)+(-1+0*i)*x*y+(-0.0335552218447942724-0.0103963159238985841*i)*(x**2)*y+(-1+0*i)*(x**3)+(-1+0*i)*(x**3)*y;', '+(-1+0*i)*(y**2)+(0.65061830020424205-0.759404916654706885*i)*x*y+(-0.00512618942425076014+0.0347528197392609484*i)*(x**2)*y+(-0.850217135729613993-0.526432162877355614*i)*(x**3)+(-0.850217135729613993-0.526432162877355614*i)*(x**3)*y;']


In [24]:
all_sols_phcpy = csolver.phcpy_solver.solve(two_eq, dictionary_output=True)

total degree : 16
2-homogeneous Bezout number : 12
  with with partition : { y }{ x }
general linear-product Bezout number : 11
  based on the set structure :
     { y }{ y x }{ x }{ x }
     { y }{ y x }{ x }{ x }
mixed volume : 4
stable mixed volume : 9


In [15]:
print(all_sols)

[[ 0.7273 -1.541j   5.0498 +0.1528j]
 [-0.8172 +1.8003j -5.0496 +0.1583j]
 [-2.9475+23.2609j -1.0015 -0.0011j]]


In [ ]:
# for j in range(4):
#     new_solver.checker_1d.GBZ_condition_check(GBZ_points[0][0], all_sols[j,new_solver.which_var],
#                                         all_sols[self.other_var], theta_detour, N_points_loop, N_points_detour, zero_eps)

In [16]:
for sol in all_sols_phcpy:
    print(sol['x'], sol['y'])

NameError: name 'all_sols_phcpy' is not defined

[[ 0.     +0.j      0.     +0.j    ]
 [-0.8172 +1.8003j -5.0496 +0.1583j]
 [ 0.7273 -1.541j   5.0498 +0.1528j]
 [-2.9475+23.2609j -1.0015 -0.0011j]]


In [17]:
print(GBZ_points[0])

[-0.0336-0.0104j  0.7273-1.541j   5.0498+0.1528j -0.8172+1.8003j]


# OBC spectrum

In [2]:
data_100x100 = sio.loadmat("data/minimal_model_OBC_spectrum_Nx_100_Ny_100.mat")
data_50x200 = sio.loadmat("data/minimal_model_OBC_spectrum_Nx_50_Ny_200.mat")
data_200x50 = sio.loadmat("data/minimal_model_OBC_spectrum_Nx_200_Ny_50.mat")

In [14]:
fig = plt.figure()
ax = fig.gca()
ax.plot(data_100x100['eigv'].real, data_100x100['eigv'].imag, '.')
ax.set_title("100 x 100")

fig = plt.figure()
ax = fig.gca()
ax.plot(data_50x200['eigv'].real, data_50x200['eigv'].imag, '.')
ax.set_title("50 x 200")
fig = plt.figure()
ax = fig.gca()
ax.plot(data_200x50['eigv'].real, data_200x50['eigv'].imag, '.')
ax.set_title("200 x 50")

Text(0.5, 1.0, '200 x 50')

In [54]:
print(char_poly.num.to_str(csolver.CStrVec(["x", "y", "z"])))

+(-1+0*i)*(z**2)+(-1+0*i)*y*z+(-1+0*i)*(y**3)+(-1+0*i)*(y**3)*z+(1+0*i)*x*(y**2)*z;


In [58]:
curr_point = non_GBZ_points[0]
print(curr_point)

[-0.0336-0.0104j  0.7273-1.541j   5.0498+0.1528j -0.8172+1.8003j]


In [60]:
eq1 = csolver.CPolyLinkedList(2)
char_poly.num.partial_eval(csolver.CScalarVec([curr_point[0]]), csolver.CIndexVec([0]), csolver.CIndexVec([1, 2]), eq1)
print(curr_point)
print(eq1.to_str(["x", "y"]))

[-0.0336-0.0104j  0.7273-1.541j   5.0498+0.1528j -0.8172+1.8003j]
+(-1+0*i)*(y**2)+(-1+0*i)*x*y+(-0.0335552218447942724-0.0103963159238985841*i)*(x**2)*y+(-1+0*i)*(x**3)+(-1+0*i)*(x**3)*y;


In [64]:
print(char_poly.num.to_str(csolver.CStrVec(['E', 'x', 'y'])))

+(-1+0*i)*(y**2)+(-1+0*i)*x*y+(-1+0*i)*(x**3)+(-1+0*i)*(x**3)*y+(1+0*i)*E*(x**2)*y;


# HN model

In [6]:
Jx1 = 1
Jx2 = 1
Jy1 = 0.8j
Jy2 = 0.8j

In [7]:
coeffs = csolver.CScalarVec([
    1, -Jx1, -Jx2, - Jy1, -Jy2
])
orders = csolver.CIntVec([
    # E, beta1, beta2
    1, 0, 0,
    0, -1, 1,
    0, 1, -1,
    0, 0, -1,
    0, 0, 1
])

In [8]:
char_poly = csolver.CLaurant(3)
char_poly.set_Laurant_by_terms(coeffs, orders)

In [9]:
char_poly.num.to_str(csolver.CStrVec(["E", "x", "y"]))

'+(-1+0*i)*(y**2)+(-0-0.800000000000000044*i)*x+(-0-0.800000000000000044*i)*x*(y**2)+(-1+0*i)*(x**2)+(1+0*i)*E*x*y;'

In [10]:
checker = csolver.DirectGBZConditionChecker(char_poly, 1)

In [15]:
phi2 = np.random.rand() * 2 * np.pi
# beta1 = np.random.rand() + 1j * np.random.rand()
beta1 = np.sqrt(Jx1 * Jy1/(Jx2 * Jy2)) * np.exp(1j * 2 * np.pi * np.random.rand())
beta2 = np.exp(1j * phi2/2) * np.sqrt((Jx2 * beta1 + Jy1)/(Jx1/beta1 + Jy2))
E = Jx1 * beta2/beta1 + Jx2 * beta1/beta2 + Jy1/ beta2 + Jy2 * beta2
res = checker.GBZ_condition_1d([E, beta1, beta2], beta1)
print(res)

True


In [18]:
print(phi2/np.pi)
print(abs(beta1))
print(beta1)

0.5240765584142799
0.9999999999999999
(-0.7351837418794432-0.6778678821688929j)


In [20]:
for curr_phi2 in np.linspace(np.pi/100, 2*np.pi*99/100, 100):
    all_sols = checker.get_aGBZ_solutions([E, beta1, beta2], [np.exp(1j*curr_phi2), np.exp(1j * curr_phi2)])
    print(all_sols)
    plt.plot(all_sols[:,0].real, all_sols[:,0].imag,'.')
plt.plot(np.abs(beta1) * np.cos(np.linspace(0, 2*np.pi, 100)), 
            np.abs(beta1) * np.sin(np.linspace(0, 2*np.pi, 100)), 
            'r-')
plt.plot(beta1.real, beta1.imag, 'x')
# plt.plot((beta1*np.exp(1j * phi2)).real, beta1.imag, 'x')
plt.show()

[[-0.2966-0.8282j  0.298 +0.2593j]
 [-0.3832+1.0702j  1.8565-1.7208j]]
[[-0.2972-0.8282j  0.3064+0.25j  ]
 [-0.3839+1.0697j  1.8005-1.7753j]]


Newton solver fails, error solution: (8.58362902089086e-32-9.40628698737033e-32j) (-1.4791141972894e-31+8.87468518373638e-31j)
The number of calls to function has reached maxfev = 500.
Newton solver fails, error solution: (3.15852510879507e-32+7.70371977754894e-33j) (-3.65926689433575e-33+4.00593428432545e-32j)
The number of calls to function has reached maxfev = 500.


[[-0.2985-0.828j   0.315 +0.2406j]
 [-0.3853+1.0688j  1.7416-1.825j ]]
[[-0.3874+1.0673j  1.6802-1.8701j]
 [-0.3005-0.8279j  0.3238+0.2311j]]
[[-0.3031-0.8276j  0.3327+0.2213j]
 [-0.3902+1.0654j  1.6166-1.9101j]]
[[-0.3937+1.063j   1.5512-1.9452j]
 [-0.3064-0.8273j  0.3419+0.2113j]]
[[-0.3105-0.8268j  0.3513+0.2011j]
 [-0.398 +1.06j    1.4842-1.975j ]]
[[-0.4031+1.0564j  1.4162-1.9996j]
 [-0.3153-0.8263j  0.3609+0.1905j]]


Newton solver fails, error solution: (6.93334779979405e-33+9.24446373305873e-33j) (-9.24446373305873e-33-6.16297582203915e-32j)
The number of calls to function has reached maxfev = 500.
Newton solver fails, error solution: (-1.38666955995881e-32+1.63704045272915e-32j) (-3.85185988877447e-34+1.42518815884655e-32j)
The number of calls to function has reached maxfev = 500.


[[-0.409 +1.0522j  1.3473-2.0189j]
 [-0.321 -0.8256j  0.3708+0.1795j]]
[[-0.3275-0.8248j  0.381 +0.1681j]
 [-0.4158+1.0473j  1.2781-2.0328j]]
[[-0.4235+1.0417j  1.2088-2.0412j]
 [-0.3349-0.8238j  0.3915+0.1562j]]
[[-0.4322+1.0352j  1.1398-2.0442j]
 [-0.3434-0.8226j  0.4024+0.1437j]]


Newton solver fails, error solution: (-2.1570415377137e-32+1.23259516440783e-32j) (-1.69481835106077e-32+1.15555796663234e-32j)
The number of calls to function has reached maxfev = 500.
Newton solver fails, error solution: (-3.22029028396467e-33-2.75423028375065e-33j) (-3.0333396624099e-33+9.05187073862001e-32j)
The number of calls to function has reached maxfev = 500.


... (52 outputs left out)

[[-0.3919+1.0643j  2.1006-1.3472j]
 [-0.3047-0.8274j  0.2543+0.3097j]]
[[-0.3017-0.8277j  0.262 +0.3005j]
 [-0.3887+1.0664j  2.0656-1.4234j]]


Newton solver fails, error solution: (3.99630463460351e-33-2.262967684655e-32j) (-7.70371977754894e-32-9.62964972193618e-34j)
The number of calls to function has reached maxfev = 500.
Newton solver fails, error solution: (2.64815367353245e-34-5.91019751683833e-33j) (-2.9274135154686e-32-7.70371977754894e-33j)
The number of calls to function has reached maxfev = 500.


[[-0.3863+1.0681j  2.0261-1.4962j]
 [-0.2994-0.828j   0.2697+0.2914j]]
[[-0.2978-0.8281j  0.2776+0.2823j]
 [-0.3845+1.0693j  1.9825-1.5653j]]
[[-0.2968-0.8282j  0.2856+0.2731j]
 [-0.3835+1.07j    1.9349-1.6305j]]


Newton solver fails, error solution: (-7.90233130306388e-33+1.27352117572606e-32j) (3.69778549322349e-32+1.01689101063646e-31j)
The number of calls to function has reached maxfev = 500.


# Number of solutions

In [4]:
char_poly = minimal_winding_char_poly()

In [39]:
print(char_poly.num.to_str(csolver.CStrVec(["E", "x", 'y'])))

+(-1+0*i)*(y**2)+(-1+0*i)*x*y+(-0.100000000000000006+0*i)*x*(y**2)+(-1+0*i)*(x**3)+(-1+0*i)*(x**3)*y+(-0.100000000000000006+0*i)*(x**4)+(1+0*i)*E*(x**2)*y;


In [6]:
curr_E = 1

In [7]:
# find all solutions
phi = np.pi/4
eff_poly1 =csolver.CPolyLinkedList(2)
char_poly.num.partial_eval(csolver.CScalarVec([curr_E]), csolver.CIndexVec([0]), csolver.CIndexVec([1,2]), eff_poly1)
eff_poly2 = eff_poly1.scale_var(csolver.CScalarVec([1, np.exp(1j * phi)]))
# eff_poly2 = eff_poly1.scale_var(csolver.CScalarVec([np.exp(1j * phi), 1]))

In [8]:
coeffs, orders = eff_poly1.batch_get_data()
print(orders)
coeffs, orders = eff_poly2.batch_get_data()
print(orders)

[[0 2]
 [1 1]
 [2 1]
 [3 0]
 [3 1]]
[[0 2]
 [1 1]
 [2 1]
 [3 0]
 [3 1]]


In [16]:
eq_list = [
    eff_poly1.to_str(['x', 'y']),
    eff_poly2.to_str(['x', 'y'])
]
print(eq_list)

['+(-1+0*i)*(y**2)+(-1+0*i)*x*y+(1+0*i)*(x**2)*y+(-1+0*i)*(x**3)+(-1+0*i)*(x**3)*y;', '+(-6.12323399573676604e-17-1*i)*(y**2)+(-0.707106781186547573-0.707106781186547462*i)*x*y+(0.707106781186547573+0.707106781186547462*i)*(x**2)*y+(-1+0*i)*(x**3)+(-0.707106781186547573-0.707106781186547462*i)*(x**3)*y;']


In [17]:
all_sols = phcpy.solver.solve(eq_list, dictionary_output=True)

total degree : 16
2-homogeneous Bezout number : 12
  with with partition : { y }{ x }
general linear-product Bezout number : 11
  based on the set structure :
     { y }{ y x }{ x }{ x }
     { y }{ y x }{ x }{ x }
mixed volume : 4
stable mixed volume : 9


In [18]:
for sol in all_sols:
    print(sol)

{'t': 0j, 'm': 5, 'err': 1.499e-17, 'rco': 0.0, 'res': 0.0, 'y': 0j, 'x': 0j}
{'t': (1+0j), 'm': 1, 'err': 1.897e-16, 'rco': 0.1192, 'res': 3.331e-16, 'y': (-0.82350905058283+1.92458739744312j), 'x': (-0.0126720721280381+1.63638127869506j)}
{'t': (1+0j), 'm': 1, 'err': 1.565e-16, 'rco': 0.1328, 'res': 8.163e-16, 'y': (-2.26876189982829+0.939751948704227j), 'x': (1.82018577615019+9.52215332468747e-17j)}
{'t': (1+0j), 'm': 1, 'err': 5.212e-16, 'rco': 0.1579, 'res': 1.943e-16, 'y': (-1.94319763375381-0.778579965682583j), 'x': (-0.012672072128038-1.63638127869506j)}
{'t': (1+0j), 'm': 1, 'err': 3.542e-17, 'rco': 0.3628, 'res': 1.393e-18, 'y': (-0.0858517593947086+0.0355609630948801j), 'x': (0.205158368105885-1.35989194645641e-17j)}


In [5]:
def solve_1d_equation(char_poly:csolver.CLaurant, curr_E, curr_beta, which_var):
    if(which_var == 1):
        other_var = 2
    else:
        other_var = 1
    eff_poly_1d = csolver.CPolyLinkedList(1)
    char_poly.num.partial_eval(csolver.CScalarVec([curr_E, curr_beta]), csolver.CIndexVec([0, which_var]),
                            csolver.CIndexVec([other_var]), eff_poly_1d)
    coeffs, degrees = eff_poly_1d.batch_get_data()
    degrees = degrees.flatten()
    max_deg = np.max(degrees)


    all_coeffs = np.zeros(max_deg + 1, dtype=complex)
    for j in range(len(degrees)):
        all_coeffs[max_deg - degrees[j]] = coeffs[j]

    all_roots = np.roots(all_coeffs)
    seq = np.argsort(np.abs(all_roots))
    return all_roots[seq]

In [6]:
solve_1d_equation(char_poly, 1, np.exp(1j*np.pi/3), 1)

array([ 1.-0.j, -1.+0.j])

In [7]:
r_arr = np.linspace(1e-3, 1, 50)
phi_arr = np.linspace(0, 2*np.pi, 50)
r_mesh, phi_mesh = np.meshgrid(r_arr, phi_arr)
beta_mesh = r_mesh * np.exp(1j * phi_mesh)

In [8]:
curr_E = 1
which_var = 1

In [9]:
if (which_var == 1):
    all_roots = np.zeros((2,) + beta_mesh.shape, dtype=complex)
else:
    all_roots = np.zeros((4,) + beta_mesh.shape, dtype=complex)

for i in range(beta_mesh.shape[0]):
    for j in range(beta_mesh.shape[1]):
        all_roots[:,i,j] = solve_1d_equation(char_poly, curr_E, beta_mesh[i,j], which_var)

In [10]:
import pyvista as pv

In [11]:
pv.set_jupyter_backend('client')

In [12]:
mesh_list = []
for j in range(all_roots.shape[0]):
    mesh_list.append(
        pv.StructuredGrid(beta_mesh.real, beta_mesh.imag, np.abs(all_roots[j,:,:]))
    )

plotter = pv.Plotter()
color = [(255,0,0), (0,0,255)]
for j in range(all_roots.shape[0]):
    plotter.add_mesh(mesh_list[j], color=color[j])
plotter.show()

Widget(value="<iframe src='http://localhost:33007/index.html?ui=P_0x7f70eda06a60_0&reconnect=auto' style='widt…

In [73]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
for j in range(2):
    ax.plot_surface(beta_mesh.real, beta_mesh.imag, np.abs(all_roots[j,:,:]))